# MyDigitalTwin — TikTok
**Notebook 05 — Ingestion, exploration, nettoyage → Parquet**

Source : `data/raw/TIKTOK/user_data_tiktok.json`

Outputs :
- `data/parquet/tiktok_watch.parquet`
- `data/parquet/tiktok_likes.parquet`
- `data/parquet/tiktok_searches.parquet`
- `data/parquet/tiktok_comments.parquet`
- `data/parquet/tiktok_messages_meta.parquet`
- `data/parquet/tiktok_messages_text.parquet`

## Objectifs ML
- **Clone NLP (axe 1)** : commentaires + messages texte
- **ALS (axe 2)** : watch history + likes
- **K-Means (axe 3)** : activité temporelle

## 0. Initialisation

In [1]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import json, hashlib

spark = SparkSession.builder \
    .appName("MyDigitalTwin - TikTok") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

RAW_PATH    = os.path.join(RAW_DATA, "TIKTOK", "user_data_tiktok.json")
MY_USERNAME = "arnvudl"  # ← ton username TikTok

def anonymize(val, salt="mydigitaltwin"):
    return hashlib.sha256(f"{salt}{val}".encode()).hexdigest()[:10]

def parse_dt(date_str):
    """Parse une date TikTok 'YYYY-MM-DD HH:MM:SS' en timestamp ms."""
    if not date_str:
        return 0
    try:
        dt = datetime.strptime(date_str.strip(), "%Y-%m-%d %H:%M:%S")
        return int(dt.replace(tzinfo=timezone.utc).timestamp() * 1000)
    except:
        return 0

def extract_video_id(url):
    """Extrait l'ID vidéo depuis une URL TikTok."""
    import re
    m = re.search(r'video/(\d+)', url or '')
    return m.group(1) if m else ""

Spark version : 3.5.5


26/04/26 00:42:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Chargement du JSON (~37 Mo)
print("Chargement du JSON TikTok...")
with open(RAW_PATH, encoding="utf-8", errors="replace") as f:
    data = json.load(f)

print("✓ Chargé — clés racine :", list(data.keys()))

Chargement du JSON TikTok...
✓ Chargé — clés racine : ['Comment', 'Direct Message', 'Income+ Wallet', 'Likes and Favorites', 'Location Review', 'Post', 'Profile And Settings', 'TikTok Live', 'TikTok Shop', 'Your Activity']


---
## PARTIE 1 — Watch History
### 1.1 Ingestion

In [3]:
raw_watch = data.get("Your Activity", {}).get("Watch History", {}).get("VideoList", [])
print(f"Vidéos regardées : {len(raw_watch):,}")

watch_rows = []
for item in raw_watch:
    ts_ms    = parse_dt(item.get("Date", ""))
    url      = item.get("Link", "")
    video_id = extract_video_id(url)
    watch_rows.append({
        "video_id":          video_id,
        "url":               url,
        "timestamp_ms":      ts_ms,
        "interaction_weight": 1.0,
        "action_type":       "watch",
        "platform":          "tiktok",
    })

schema_watch = StructType([
    StructField("video_id",           StringType(), True),
    StructField("url",                StringType(), True),
    StructField("timestamp_ms",       LongType(),   True),
    StructField("interaction_weight", DoubleType(), True),
    StructField("action_type",        StringType(), True),
    StructField("platform",           StringType(), True),
])

df_watch = spark.createDataFrame(watch_rows, schema=schema_watch) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_watch.count():,} lignes")
df_watch.show(5, truncate=60)

Vidéos regardées : 234,593


26/04/26 00:42:44 WARN TaskSetManager: Stage 0 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


DataFrame : 234,593 lignes


26/04/26 00:42:48 WARN TaskSetManager: Stage 3 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|           video_id|                                                     url| timestamp_ms|interaction_weight|action_type|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|7405385487155776811|https://www.tiktokv.com/share/video/7405385487155776811/|1761745244000|               1.0|      watch|  tiktok|2025-10-29 13:40:44|      2025|    2025-10|        13|            4|
|7511397803344629014|https://www.tiktokv.com/share/video/7511397803344629014/|1761745257000|               1.0|      watch|  tiktok|2025-10-29 13:40:57|      2025|    2025-10|        13|          

### 1.2 Exploration

In [4]:
print("=== Vidéos regardées par année ===")
df_watch.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_watch.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Mois les plus actifs ===")
df_watch.groupBy("event_month") \
    .count().orderBy(F.desc("count")).limit(10).show()

=== Vidéos regardées par année ===


26/04/26 00:42:58 WARN TaskSetManager: Stage 4 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.
26/04/26 00:43:00 WARN TaskSetManager: Stage 7 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+----------+------+
|event_year| count|
+----------+------+
|      2025|173383|
|      2026| 61210|
+----------+------+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|11511|
|         1| 8939|
|         2| 5270|
|         3| 1624|
|         4|  389|
|         5| 2645|
|         6| 3582|
|         7| 4562|
|         8| 8819|
|         9|11417|
|        10|15344|
|        11|13132|
|        12|11503|
|        13| 7331|
|        14| 8299|
|        15|11112|
|        16|11573|
|        17|10702|
|        18|16328|
|        19|15923|
+----------+-----+
only showing top 20 rows


=== Mois les plus actifs ===


26/04/26 00:43:00 WARN TaskSetManager: Stage 10 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+-----------+-----+
|event_month|count|
+-----------+-----+
|    2025-12|22328|
|    2026-01|18242|
|    2026-02|17908|
|    2025-05|15709|
|    2025-08|15695|
|    2025-03|15664|
|    2025-04|15496|
|    2025-07|15345|
|    2025-11|15177|
|    2025-09|14819|
+-----------+-----+



### 1.3 Écriture Parquet

In [5]:
df_watch.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_watch"))
print(f"tiktok_watch -- {df_watch.count():,} lignes")

26/04/26 00:43:06 WARN TaskSetManager: Stage 13 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.
26/04/26 00:43:14 WARN TaskSetManager: Stage 15 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


tiktok_watch -- 234,593 lignes


---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [6]:
raw_likes = data.get("Likes and Favorites", {}).get("Like List", {}).get("ItemFavoriteList", [])
print(f"Likes : {len(raw_likes):,}")

like_rows = []
for item in raw_likes:
    ts_ms    = parse_dt(item.get("date", ""))
    url      = item.get("link", "")
    video_id = extract_video_id(url)
    like_rows.append({
        "video_id":           video_id,
        "url":                url,
        "timestamp_ms":       ts_ms,
        "interaction_weight": 2.0,
        "action_type":        "like",
        "platform":           "tiktok",
    })

df_likes = spark.createDataFrame(like_rows, schema=schema_watch) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_likes.count():,} lignes")
df_likes.show(5, truncate=60)

Likes : 6,000
DataFrame : 6,000 lignes
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|           video_id|                                                     url| timestamp_ms|interaction_weight|action_type|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|7620909048401054998|https://www.tiktokv.com/share/video/7620909048401054998/|1777038133000|               2.0|       like|  tiktok|2026-04-24 13:42:13|      2026|    2026-04|        13|            6|
|7632317404705525014|https://www.tiktokv.com/share/video/7632317404705525014/|1777038057000|               2.0|       like|  tiktok|2026-04-24 13:40:57|     

### 2.2 Exploration

In [7]:
print("=== Likes par année ===")
df_likes.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_likes.groupBy("event_hour").count().orderBy("event_hour").show()

=== Likes par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2026| 6000|
+----------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  269|
|         1|  400|
|         2|  245|
|         3|   79|
|         5|   47|
|         6|  122|
|         7|   60|
|         8|   71|
|         9|  197|
|        10|  161|
|        11|  413|
|        12|  331|
|        13|   66|
|        14|  224|
|        15|  192|
|        16|  254|
|        17|  220|
|        18|  602|
|        19|  567|
|        20|  136|
+----------+-----+
only showing top 20 rows



### 2.3 Écriture Parquet

In [8]:
df_likes.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_likes"))
print(f"tiktok_likes -- {df_likes.count():,} lignes")

tiktok_likes -- 6,000 lignes


---
## PARTIE 2B — Vidéos sauvegardées (Saves)
### 2B.1 Ingestion

In [ ]:
raw_saves = data.get("Likes and Favorites", {}).get("Favorite Videos", {}).get("FavoriteVideoList", [])
print(f"Saves : {len(raw_saves):,}")

save_rows = []
for item in raw_saves:
    ts_ms    = parse_dt(item.get("Date", ""))
    url      = item.get("Link", "")
    video_id = extract_video_id(url)
    save_rows.append({
        "video_id":           video_id,
        "url":                url,
        "timestamp_ms":       ts_ms,
        "interaction_weight": 3.0,
        "action_type":        "save",
        "platform":           "tiktok",
    })

df_saves = spark.createDataFrame(save_rows, schema=schema_watch) 
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) 
    .withColumn("event_year",    F.year("event_date")) 
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) 
    .withColumn("event_hour",    F.hour("event_date")) 
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_saves.count():,} lignes")
df_saves.show(5, truncate=60)

### 2B.2 Écriture Delta

In [ ]:
df_saves.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_saves"))
print(f"tiktok_saves -- {df_saves.count():,} lignes")

---
## PARTIE 3 — Recherches
### 3.1 Ingestion

In [9]:
raw_searches = data.get("Your Activity", {}).get("Searches", {}).get("SearchList", [])
print(f"Recherches : {len(raw_searches):,}")

search_rows = []
for item in raw_searches:
    ts_ms = parse_dt(item.get("Date", ""))
    query = item.get("SearchTerm", "")
    search_rows.append({
        "query":        query,
        "timestamp_ms": ts_ms,
        "char_count":   len(query),
        "word_count":   len(query.split()),
        "platform":     "tiktok",
    })

schema_search = StructType([
    StructField("query",        StringType(), True),
    StructField("timestamp_ms", LongType(),   True),
    StructField("char_count",   IntegerType(), True),
    StructField("word_count",   IntegerType(), True),
    StructField("platform",     StringType(), True),
])

df_searches = spark.createDataFrame(search_rows, schema=schema_search) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_searches.count():,} lignes")
df_searches.show(10, truncate=50)

Recherches : 1,122
DataFrame : 1,122 lignes
+-----------------------------+-------------+----------+----------+--------+-------------------+----------+-----------+----------+-------------+
|                        query| timestamp_ms|char_count|word_count|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-----------------------------+-------------+----------+----------+--------+-------------------+----------+-----------+----------+-------------+
|              anyme bb jacque|1761745215000|        15|         3|  tiktok|2025-10-29 13:40:15|      2025|    2025-10|        13|            4|
|          free excel tempkate|1761745229000|        19|         3|  tiktok|2025-10-29 13:40:29|      2025|    2025-10|        13|            4|
|          free excel template|1761745233000|        19|         3|  tiktok|2025-10-29 13:40:33|      2025|    2025-10|        13|            4|
|          free excel tempkate|1761745271000|        19|         3|  tiktok|2025-10-29

### 3.2 Exploration

In [10]:
print("=== Top 20 termes recherchés ===")
df_searches.groupBy("query") \
    .count().orderBy(F.desc("count")).limit(20).show(truncate=50)

print("\n=== Top 20 mots recherchés ===")
df_searches \
    .withColumn("word", F.explode(F.split(F.lower("query"), r"\s+"))) \
    .filter(F.length("word") > 1) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Recherches par année ===")
df_searches.groupBy("event_year").count().orderBy("event_year").show()

=== Top 20 termes recherchés ===
+------------------------------+-----+
|                         query|count|
+------------------------------+-----+
|                           PGE|    5|
|                      anyme023|    5|
|                  playboicarti|    5|
|freed from desire x woops edit|    4|
|                       Maxence|    4|
|                    Malien🇫🇷|    4|
|           pile version gospel|    4|
|               anyme bb jacque|    3|
|                    92 et puis|    3|
|   Jsuis dqns le bat a djibril|    3|
|                       Et puis|    3|
|              chant alcoolique|    3|
|                        funana|    3|
|                Pack sample dj|    3|
|    Cosplay vladimir cauchemar|    3|
|     tas a tocar bue mas mesmo|    3|
|           big mama transition|    3|
|                    911 riddim|    3|
|                marceau bateau|    2|
|              evil jordan edit|    2|
+------------------------------+-----+


=== Top 20 mots recherchés ===


### 3.3 Écriture Parquet

In [11]:
df_searches.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_searches"))
print(f"tiktok_searches -- {df_searches.count():,} lignes")

tiktok_searches -- 1,122 lignes


---
## PARTIE 4 — Commentaires
### 4.1 Ingestion

In [12]:
raw_comments = data.get("Comment", {}).get("Comments", {}).get("CommentsList", [])
print(f"Commentaires : {len(raw_comments):,}")

comment_rows = []
for item in raw_comments:
    ts_ms   = parse_dt(item.get("date", ""))
    comment = item.get("comment", "")
    url     = item.get("url", "")
    comment_rows.append({
        "text":         comment,
        "url":          url,
        "timestamp_ms": ts_ms,
        "char_count":   len(comment),
        "word_count":   len(comment.split()),
        "platform":     "tiktok",
        "content_type": "comment",
    })

schema_comments = StructType([
    StructField("text",         StringType(),  True),
    StructField("url",          StringType(),  True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("char_count",   IntegerType(), True),
    StructField("word_count",   IntegerType(), True),
    StructField("platform",     StringType(),  True),
    StructField("content_type", StringType(),  True),
])

df_comments = spark.createDataFrame(comment_rows, schema=schema_comments) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_comments.count():,} lignes")
df_comments.show(5, truncate=60)

Commentaires : 635
DataFrame : 635 lignes
+------------------------------------------------------------+---+-------------+----------+----------+--------+------------+-------------------+----------+-----------+----------+-------------+
|                                                        text|url| timestamp_ms|char_count|word_count|platform|content_type|         event_date|event_year|event_month|event_hour|event_weekday|
+------------------------------------------------------------+---+-------------+----------+----------+--------+------------+-------------------+----------+-----------+----------+-------------+
|          C quoi ça ?? Comment ça il est mondial le p'tit ??|   |1772115558000|        50|        12|  tiktok|     comment|2026-02-26 14:19:18|      2026|    2026-02|        14|            5|
|C'est pas obligatoirement une image, NFT c'est pour jeton...|   |1770433090000|       120|        20|  tiktok|     comment|2026-02-07 02:58:10|      2026|    2026-02|         2|        

### 4.2 Exploration

In [13]:
print("=== Stats texte ===")
df_comments.agg(
    F.avg("char_count").alias("avg_chars"),
    F.avg("word_count").alias("avg_words"),
    F.max("char_count").alias("max_chars"),
).show()

print("\n=== Top 20 mots ===")
df_comments \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Commentaires par année ===")
df_comments.groupBy("event_year").count().orderBy("event_year").show()

=== Stats texte ===
+-----------------+-----------------+---------+
|        avg_chars|        avg_words|max_chars|
+-----------------+-----------------+---------+
|33.69133858267717|6.823622047244094|      150|
+-----------------+-----------------+---------+


=== Top 20 mots ===
+-----+-----+
| word|count|
+-----+-----+
|  pas|   89|
|  que|   53|
| mais|   45|
|  des|   43|
|  les|   42|
|c'est|   39|
|  est|   36|
| pour|   34|
| fait|   27|
|  qui|   26|
| j'ai|   25|
| dans|   24|
|  une|   24|
|  sur|   24|
|  dit|   23|
|  moi|   20|
|  son|   19|
|faire|   19|
|aussi|   18|
|  non|   17|
+-----+-----+


=== Commentaires par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2019|    1|
|      2020|  135|
|      2021|  397|
|      2022|   66|
|      2023|    6|
|      2024|    2|
|      2025|   24|
|      2026|    4|
+----------+-----+



### 4.3 Écriture Parquet

In [14]:
df_comments.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_comments"))
print(f"tiktok_comments -- {df_comments.count():,} lignes")

tiktok_comments -- 635 lignes


---
## PARTIE 5 — Messages (métadonnées + texte)

> **Deux Parquets** :
> - `tiktok_messages_meta.parquet` — anonymisé, pour K-Means temporel
> - `tiktok_messages_text.parquet` — tes messages uniquement, pour NLP Clone

### 5.1 Ingestion

In [15]:
chat_history = data.get("Direct Message", {}) \
                   .get("Direct Messages", {}) \
                   .get("ChatHistory", {})

print(f"Conversations : {len(chat_history):,}")

meta_rows = []
text_rows = []

for conv_key, messages in chat_history.items():
    # Extraire le nom de l'interlocuteur depuis la clé
    # ex: "Chat History with dj_yoyo206:"
    import re
    match = re.search(r'with\s+(.+?):', conv_key)
    other = match.group(1) if match else conv_key
    conv_id = anonymize(conv_key)

    if not isinstance(messages, list):
        continue

    for msg in messages:
        ts_ms   = parse_dt(msg.get("Date", ""))
        sender  = msg.get("From", "")
        content = msg.get("Content", "")
        is_me   = sender == MY_USERNAME

        # Détecter le type
        has_url = content.startswith("http") if content else False
        msg_type = "url" if has_url else ("text" if content else "other")

        # Métadonnées
        meta_rows.append({
            "conv_id":      conv_id,
            "sender_anon":  anonymize(sender),
            "is_me":        is_me,
            "timestamp_ms": ts_ms,
            "msg_type":     msg_type,
            "char_count":   len(content) if content else 0,
            "platform":     "tiktok",
        })

        # Texte — uniquement mes messages non-URL
        if is_me and content and not has_url:
            text_rows.append({
                "text":         content,
                "timestamp_ms": ts_ms,
                "platform":     "tiktok",
                "content_type": "dm",
            })

print(f"Messages (métadonnées) : {len(meta_rows):,}")
print(f"Mes messages (texte)   : {len(text_rows):,}")

Conversations : 12
Messages (métadonnées) : 12,044
Mes messages (texte)   : 4,374


In [16]:
# DataFrame métadonnées
schema_meta = StructType([
    StructField("conv_id",      StringType(),  True),
    StructField("sender_anon",  StringType(),  True),
    StructField("is_me",        BooleanType(), True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("msg_type",     StringType(),  True),
    StructField("char_count",   IntegerType(), True),
    StructField("platform",     StringType(),  True),
])

df_meta = spark.createDataFrame(meta_rows, schema=schema_meta) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

# DataFrame texte
schema_text = StructType([
    StructField("text",         StringType(), True),
    StructField("timestamp_ms", LongType(),   True),
    StructField("platform",     StringType(), True),
    StructField("content_type", StringType(), True),
])

df_text = spark.createDataFrame(text_rows, schema=schema_text) \
    .withColumn("event_date",  F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("char_count",  F.length("text")) \
    .withColumn("word_count",  F.size(F.split(F.trim("text"), r"\s+")))

print(f"Métadonnées : {df_meta.count():,} lignes")
print(f"Texte       : {df_text.count():,} lignes")
df_meta.show(5)
df_text.show(5, truncate=60)

Métadonnées : 12,044 lignes
Texte       : 4,374 lignes
+----------+-----------+-----+-------------+--------+----------+--------+-------------------+----------+-----------+----------+-------------+
|   conv_id|sender_anon|is_me| timestamp_ms|msg_type|char_count|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+----------+-----------+-----+-------------+--------+----------+--------+-------------------+----------+-----------+----------+-------------+
|433529bbab| 602455d85e| true|1776969115000|     url|        56|  tiktok|2026-04-23 18:31:55|      2026|    2026-04|        18|            5|
|433529bbab| 602455d85e| true|1774729867000|     url|        56|  tiktok|2026-03-28 20:31:07|      2026|    2026-03|        20|            7|
|433529bbab| 020053daa3|false|1774527887000|     url|        56|  tiktok|2026-03-26 12:24:47|      2026|    2026-03|        12|            5|
|433529bbab| 602455d85e| true|1774378012000|     url|        56|  tiktok|2026-03-24 18:46:52|

### 5.2 Exploration

In [17]:
print("=== Répartition par type de message ===")
df_meta.groupBy("msg_type").count().orderBy(F.desc("count")).show()

print("\n=== Moi vs les autres ===")
df_meta.groupBy("is_me").count().show()

print("\n=== Activité par heure ===")
df_meta.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top conversations (nb messages) ===")
df_meta.groupBy("conv_id") \
    .count().orderBy(F.desc("count")).limit(10).show()

print("\n=== Top 20 mots (mes messages) ===")
df_text \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Répartition par type de message ===
+--------+-----+
|msg_type|count|
+--------+-----+
|    text| 8707|
|     url| 3337|
+--------+-----+


=== Moi vs les autres ===
+-----+-----+
|is_me|count|
+-----+-----+
| true| 6214|
|false| 5830|
+-----+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  391|
|         1|  458|
|         2|  104|
|         3|   17|
|         4|   12|
|         5|   57|
|         6|   94|
|         7|  252|
|         8|  488|
|         9|  658|
|        10|  870|
|        11|  602|
|        12|  626|
|        13|  441|
|        14|  378|
|        15|  566|
|        16|  461|
|        17|  482|
|        18|  850|
|        19|  791|
+----------+-----+
only showing top 20 rows


=== Top conversations (nb messages) ===
+----------+-----+
|   conv_id|count|
+----------+-----+
|c45362e49b| 5997|
|7cc9af6c53| 2370|
|3e11724c44| 1470|
|8b0a991826|  905|
|3c7b1bbf91|  617|
|3801e80092|  416|
|433529bbab|  145|
|dfa

### 5.3 Écriture Parquet

In [18]:
df_meta.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_messages_meta"))
print(f"tiktok_messages_meta -- {df_meta.count():,} lignes")

df_text.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_messages_text"))
print(f"tiktok_messages_text -- {df_text.count():,} lignes")

tiktok_messages_meta -- 12,044 lignes
tiktok_messages_text -- 4,374 lignes


In [19]:
# -- PARTIE 6 - Hashtags favoris --
fav_hashtags = data.get("Likes and Favorites", {}).get("Favorite Hashtags", {}).get("FavoriteHashtagList", [])
print(f"Hashtags favoris : {len(fav_hashtags):,}")

ht_rows = []
for ht in fav_hashtags:
    hashtag = ht.get("HashtagName", "") or ht.get("hashtag", "")
    if hashtag:
        ht_rows.append({"hashtag": hashtag, "platform": "tiktok"})

if ht_rows:
    from pyspark.sql.types import StructType, StructField, StringType as _ST
    schema_ht = StructType([StructField("hashtag", _ST(), True), StructField("platform", _ST(), True)])
    df_ht = spark.createDataFrame(ht_rows, schema=schema_ht)
    df_ht.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_favorite_hashtags"))
    print(f"tiktok_favorite_hashtags -- {df_ht.count():,} lignes")
else:
    print("Aucun hashtag favori trouve")

# -- PARTIE 7 - Centres d'interet publicitaires --
ads_raw = data.get("Ads", {}).get("Interests", [])
if isinstance(ads_raw, dict):
    ads_raw = ads_raw.get("adInterestCategories", [])
print(f"Centres d'interet : {len(ads_raw):,}")

ai_rows = []
for item in ads_raw:
    if isinstance(item, str):
        cat = item
    else:
        cat = item.get("Name", "") or item.get("category", "")
    if cat:
        ai_rows.append({"category": cat, "platform": "tiktok"})

if ai_rows:
    from pyspark.sql.types import StructType, StructField, StringType as _ST
    schema_ai = StructType([StructField("category", _ST(), True), StructField("platform", _ST(), True)])
    df_ai = spark.createDataFrame(ai_rows, schema=schema_ai)
    df_ai.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "tiktok_ads_interests"))
    print(f"tiktok_ads_interests -- {df_ai.count():,} lignes")
else:
    print("Aucun centre d'interet publicitaire trouve")

Hashtags favoris : 0
Aucun hashtag favori trouve
Centres d'interet : 0
Aucun centre d'interet publicitaire trouve


In [20]:
spark.stop()